# Recurrent Neural Networks

**Companion lesson:** https://ml-viz.vercel.app/courses/rnns/01-recurrent-neural-networks

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A vanilla RNN cell, from scratch

$h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1} + b_h)$, with the same weights reused at every step. We build the forward pass and run it over a sequence.

In [ ]:
class VanillaRNN:
    def __init__(self, n_in, n_hidden, n_out, scale=0.1):
        self.Wxh = np.random.randn(n_hidden, n_in) * scale
        self.Whh = np.random.randn(n_hidden, n_hidden) * scale
        self.Why = np.random.randn(n_out, n_hidden) * scale
        self.bh = np.zeros((n_hidden, 1))
        self.by = np.zeros((n_out, 1))
        self.n_hidden = n_hidden

    def forward(self, xs):
        """xs: list of (n_in, 1) column vectors. Returns outputs and hidden states."""
        h = np.zeros((self.n_hidden, 1))
        hs, ys = [h], []
        for x in xs:
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
            y = self.Why @ h + self.by
            hs.append(h); ys.append(y)
        return ys, hs

rnn = VanillaRNN(n_in=3, n_hidden=5, n_out=2)
seq = [np.random.randn(3, 1) for _ in range(6)]
ys, hs = rnn.forward(seq)
print('processed', len(seq), 'steps')
print('final hidden state:', np.round(hs[-1].ravel(), 3))
print('final output:', np.round(ys[-1].ravel(), 3))

## Worked forward pass — two timesteps, verified by hand

The lesson runs this exact example. With $D=2$, $H=2$, $O=1$ and tiny explicit weights

$$W_{xh}=\begin{bmatrix}0.5 & -0.3\\ 0.1 & 0.4\end{bmatrix},\; W_{hh}=0.2 I,\; W_{hy}=\begin{bmatrix}1 & -1\end{bmatrix},\; \mathbf b_h=\mathbf 0,\; b_y=0,$$

feed $\mathbf x_1=[1,2]^\top$ then $\mathbf x_2=[-1,1]^\top$ from $\mathbf h_0=\mathbf 0$. We compute it with pure stdlib `math.tanh` (no numpy needed) so the output is deterministic, and `assert` the hand-derived numbers.

In [ ]:
import math

Wxh = [[0.5, -0.3], [0.1, 0.4]]   # H x D
Whh = [[0.2, 0.0], [0.0, 0.2]]    # H x H
Why = [1.0, -1.0]                 # O x H (single output row)

def matvec(M, v):
    return [sum(M[i][j] * v[j] for j in range(len(v))) for i in range(len(M))]

def rnn_step(x, h):
    a = matvec(Wxh, x)            # W_xh x_t
    b = matvec(Whh, h)            # W_hh h_{t-1}
    pre = [a[i] + b[i] for i in range(2)]   # + b_h (zero)
    h_new = [math.tanh(p) for p in pre]
    y = sum(Why[i] * h_new[i] for i in range(2))   # W_hy h_t + b_y (zero)
    return pre, h_new, y

h0 = [0.0, 0.0]
pre1, h1, y1 = rnn_step([1.0, 2.0], h0)
pre2, h2, y2 = rnn_step([-1.0, 1.0], h1)

print("h1 =", [round(v, 4) for v in h1], " y1 =", round(y1, 4))
print("h2 =", [round(v, 4) for v in h2], " y2 =", round(y2, 4))

# Hand-derived values from the lesson (full precision through the recurrence)
assert [round(v, 4) for v in h1] == [-0.0997, 0.7163]
assert round(y1, 4) == -0.816
assert [round(v, 4) for v in h2] == [-0.675, 0.4163]
assert round(y2, 4) == -1.0914
print("verified: forward pass matches the by-hand derivation")

## Parameter count is independent of sequence length

Weight sharing means the cell has $HD + H^2 + H$ parameters and the output head adds $OH + O$ — no dependence on how many timesteps you run. We tally them for $D=3, H=4, O=2$ and confirm the total is **42**.

In [ ]:
def rnn_param_count(D, H, O):
    Wxh = H * D
    Whh = H * H
    bh = H
    Why = O * H
    by = O
    cell = Wxh + Whh + bh          # the recurrent part reused every step
    total = cell + Why + by
    return Wxh, Whh, bh, Why, by, cell, total

Wxh, Whh, bh, Why, by, cell, total = rnn_param_count(3, 4, 2)
print(f"Wxh={Wxh}  Whh={Whh}  bh={bh}  Why={Why}  by={by}")
print(f"recurrent cell only = {cell}")
print(f"total parameters    = {total}")

assert (Wxh, Whh, bh, Why, by) == (12, 16, 4, 8, 2)
assert cell == 32 and total == 42
print("verified: counts match the lesson (cell 32, total 42)")

## The hidden state accumulates context

Feed a constant input and watch a single hidden unit build up over time.

In [ ]:
Wxh, Whh = 0.5, 0.9
h, hist = 0.0, []
for t in range(15):
    h = np.tanh(Wxh * 1.0 + Whh * h)
    hist.append(h)
plt.plot(range(1, 16), hist, 'o-', color='#6366f1')
plt.xlabel('time step'); plt.ylabel('hidden state'); plt.title('Memory accumulates, then saturates')
plt.show()

## A real task: sequence parity

Can an RNN learn to output whether the number of 1s seen so far is odd? This needs **memory** — the answer depends on the whole prefix, not the current bit. We train the tiny RNN above with BPTT (next lesson covers the gradients in detail).

In [ ]:
def make_parity(T, n):
    X = np.random.randint(0, 2, size=(n, T))
    Y = np.cumsum(X, axis=1) % 2          # running parity at each step
    return X, Y

def sigmoid(z): return 1 / (1 + np.exp(-z))

def train_parity(T=8, epochs=4000, lr=0.2, n_hidden=16):
    nh = n_hidden
    Wxh = np.random.randn(nh, 1) * 0.3
    Whh = np.random.randn(nh, nh) * 0.3
    Why = np.random.randn(1, nh) * 0.3
    bh, by = np.zeros((nh, 1)), np.zeros((1, 1))
    losses = []
    for ep in range(epochs):
        X, Y = make_parity(T, 32)
        gWxh = gWhh = gWhy = gbh = gby = 0
        loss = 0
        for b in range(X.shape[0]):
            hs = [np.zeros((nh, 1))]; ps = []; xs = []
            for t in range(T):
                x = np.array([[X[b, t]]], float); xs.append(x)
                h = np.tanh(Wxh @ x + Whh @ hs[-1] + bh); hs.append(h)
                p = sigmoid(Why @ h + by); ps.append(p)
                loss += -(Y[b, t]*np.log(p+1e-9) + (1-Y[b, t])*np.log(1-p+1e-9)).item()
            dWxh=dWhh=dWhy=dbh=dby=0; dh_next=np.zeros((nh,1))
            for t in reversed(range(T)):
                dy = ps[t] - Y[b, t]
                dWhy += dy @ hs[t+1].T; dby += dy
                dh = Why.T @ dy + dh_next
                draw = (1 - hs[t+1]**2) * dh
                dWxh += draw @ xs[t].T; dWhh += draw @ hs[t].T; dbh += draw
                dh_next = Whh.T @ draw
            gWxh+=dWxh; gWhh+=dWhh; gWhy+=dWhy; gbh+=dbh; gby+=dby
        m = X.shape[0]
        for p,g in [(Wxh,gWxh),(Whh,gWhh),(Why,gWhy),(bh,gbh),(by,gby)]:
            np.clip(g, -5, 5, out=g); p -= lr * g / m
        losses.append(loss / (m*T))
    return losses

losses = train_parity(T=8)
plt.plot(losses, color='#14b8a6'); plt.xlabel('epoch'); plt.ylabel('cross-entropy')
plt.title('Vanilla RNN learns 8-step parity'); plt.show()
print('final loss:', round(losses[-1], 4))

## Key takeaways

- An RNN reuses one set of weights across time, carrying a hidden state as memory.
- The forward pass is a simple `tanh` recurrence; training uses BPTT (next lesson).
- It can genuinely learn order-dependent tasks like running parity.
- Gradient clipping (`np.clip`) keeps training stable.